# Check the Policy Summary

| Item | Description |
|---|---|
| Goal | Read the T3.5.5 run and write down what it produced: the settings and calls, every per-year summary, the final policy summary with its token count, and a trace of the summary's specific details back to the source statements. No model calls. |
| Input | `output/T3.5.5/policy_summary.md`, `policy_summary_meta.json`, `yearly_summaries.csv`; the call log on scratch; `output/T3.5.2/topic68_statements.csv` for the source trace. |
| Output | `output/T3.5.6/policy_summary_check.md`, a readable record of all of the above. |
| Unit | One run of T3.5.5, identified by its start time in the metadata. |
| Checks | The final summary is at most 512 bge tokens; the run made one map call per year and at least one reduce call, none hitting the output cap; each specific detail named in the summary either appears in at least one statement or is flagged as unsourced. |

See `Topic Relevance Rebuild Plan.md` in the TEMP folder for how the summary is used downstream.


## Step 1. Import packages

Read the run files and display them.


In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


## Step 2. Define file locations and the details to trace

The trace terms are the specific requirements, actors, and data types the final summary names, written down by reading it. A term with no matching statement is a detail the model introduced.


In [2]:
PROJECT_FOLDER = Path("/project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis")
NOTEBOOK_SUBFOLDER = Path("T3 - Topic Relevance and Corporate Disclosure/TEMP")
NOTEBOOK_NAME = "T3.5.6"  # the identifier names this notebook's output folder
RUN_FOLDER = PROJECT_FOLDER / NOTEBOOK_SUBFOLDER / "output/T3.5.5"
STATEMENTS_FILE = PROJECT_FOLDER / NOTEBOOK_SUBFOLDER / "output/T3.5.2/topic68_statements.csv"
OUTPUT_FOLDER = PROJECT_FOLDER / NOTEBOOK_SUBFOLDER / "output" / NOTEBOOK_NAME

MAX_BGE_TOKENS = 512  # the encoder's input limit the final summary must respect
TRACE_TERMS = [  # specific details named in the final summary, read off the text
    "broadband", "healthcare", "drone", "financial institution", "opt in", "opt out",
    "clear and conspicuous", "firewall", "privacy by design", "72 hours", "health record",
    "social security number", "geolocation", "discrimination", "identity theft", "monitoring",
    "algorithm", "research center", "aggregator", "federal agenc", "export", "directory information",
    "voice recording", "connected vehicle", "national security", "uniform national",
]
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"Outputs: {OUTPUT_FOLDER}")


Outputs: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.5.6


## Step 3. Load the run metadata and the final summary

The metadata records the model, settings, GPU, and timing; the summary must fit the encoder's limit.


In [3]:
meta = json.loads((RUN_FOLDER / "policy_summary_meta.json").read_text(encoding="utf-8"))
policy_summary = (RUN_FOLDER / "policy_summary.md").read_text(encoding="utf-8").strip()
assert meta["final_n_bge_tokens"] <= MAX_BGE_TOKENS
assert len(policy_summary.split()) == meta["final_n_words"]
print(f"Model: {Path(meta['model']).parent.parent.name}; GPU: {meta['gpu']}; generation: {meta['generation']}; thinking: {meta['enable_thinking']}; seed: {meta['seed']}")
print(f"Run: {meta['run_started']} to {meta['run_finished']}; map calls: {meta['n_map_calls']}; reduce rounds: {len(meta['reduce_rounds'])}")
print(f"Final summary: {meta['final_n_words']} words, {meta['final_n_bge_tokens']} bge tokens (limit {MAX_BGE_TOKENS})")


Model: models--Qwen--Qwen3.5-4B; GPU: NVIDIA A40; generation: {'do_sample': True, 'temperature': 0.1}; thinking: False; seed: 0
Run: 2026-10-25T12:37:24+00:00 to 2026-10-25T12:40:20+00:00; map calls: 26; reduce rounds: 1
Final summary: 221 words, 274 bge tokens (limit 512)


## Step 4. Load the call log for this run

The log on scratch holds every prompt and completion; keep the calls of this run and tabulate them.


In [4]:
calls = pd.read_json(meta["call_log"], lines=True)
calls = calls.loc[calls["run_started"].eq(meta["run_started"])].reset_index(drop=True)
assert calls["stage"].eq("map").sum() == meta["n_map_calls"]
assert calls["stage"].eq("reduce").sum() == len(meta["reduce_rounds"])
assert not calls["hit_cap"].any()
call_table = calls[["stage", "label", "n_input_tokens", "n_output_tokens", "seconds", "hit_cap"]]
print(f"Calls in this run: {len(calls)}; total input tokens {calls['n_input_tokens'].sum():,}; total seconds {calls['seconds'].sum():.0f}")
display(call_table)


Calls in this run: 27; total input tokens 30,762; total seconds 176


,stage,label,n_input_tokens,n_output_tokens,seconds,hit_cap
0,map,1999,683,164,8.9,False
1,map,2000,1666,118,5.3,False
2,map,2001,1100,171,7.4,False
3,map,2002,1350,135,6.0,False
4,map,2003,648,122,5.3,False
5,map,2004,547,132,5.7,False
6,map,2005,346,111,4.8,False
7,map,2006,606,128,5.5,False
8,map,2007,281,114,4.9,False
9,map,2008,612,141,6.0,False


## Step 5. Show the per-year summaries

One summary per hearing year with statements. Years with one or two statements were asked to write only a sentence or two; their word counts show whether the model obeyed.


In [5]:
yearly = pd.read_csv(RUN_FOLDER / "yearly_summaries.csv")
assert len(yearly) == meta["n_map_calls"]
assert yearly["n_bge_tokens"].le(MAX_BGE_TOKENS).all()
display(yearly[["hearing_year", "n_statements", "n_words", "n_bge_tokens", "hit_cap"]].set_index("hearing_year").T)
small_years = yearly.loc[yearly["n_statements"].le(2)]
print(f"Years with at most two statements: {small_years['hearing_year'].tolist()}; their summary lengths in words: {small_years['n_words'].tolist()}")
yearly_markdown = "\n\n".join(
    f"**{row.hearing_year}** ({row.n_statements} statements, {row.n_words} words): {row.summary}"
    for row in yearly.itertuples(index=False)
)
display(Markdown(yearly_markdown))


hearing_year,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
n_statements,9,33,15,13,8,4,2,8,1,6,...,21,11,44,40,6,62,1,10,9,6
n_words,141,104,143,116,100,117,89,108,101,119,...,133,132,129,143,108,120,94,124,122,124
n_bge_tokens,170,117,176,140,127,131,111,132,117,145,...,162,150,168,181,125,152,113,144,139,142
hit_cap,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


Years with at most two statements: [2005, 2007, 2022]; their summary lengths in words: [89, 101, 94]


**1999** (9 statements, 141 words): The discussed legislation seeks to establish guarantees and assurances that business and personal information remains confidential as the Internet expands into global commerce. It specifically addresses the export of products generally available to the public, granting the administration a one-time review period before such exports are permitted. Furthermore, the proposed PROTECT Act aims to explicitly delineate the rights and procedural protections for Americans regarding their ability to use encryption and remain secure with encrypted data. The legislation attempts to balance legitimate law enforcement interests with individual privacy rights by setting specific procedures for this balance. Additionally, it targets health Web sites to require disclosure of commercial interests, aiming to help the public distinguish between nonprofit-operated sites and those driven by profit motives. Overall, the regulations focus on data confidentiality, encryption usage, export controls, and transparency regarding commercial interests on health-related websites.

**2000** (33 statements, 104 words): The proposed legislation seeks to regulate companies and organizations that make a business of collecting personally identifiable information online. It aims to establish a baseline standard for consumer privacy, moving beyond self-regulation and voluntary privacy policies which are often deemed insufficient or overly complex. The law targets the gathering and dissemination of personal data, specifically addressing practices where information is collected without consumer knowledge or where trust is violated. By introducing strong enforcement mechanisms to punish bad actors, the legislation intends to ensure minimum consumer privacy protection in the digital era, preventing a scenario where the lack of regulation erodes public confidence in technology.

**2001** (15 statements, 143 words): The proposed legislation seeks to regulate companies and organizations handling personal information by establishing meaningful privacy protections to prevent a fragmented landscape of 50 different standards. It specifically targets the collection, use, and disclosure of data, aiming to balance consumer choice—allowing individuals to opt in or out—with the need for security and e-commerce stability. The regulations distinguish between data types, asserting that financial and medical information deserve the highest level of privacy and sufficient protection, while also addressing concerns about customized services and online profiling. The legislation intends to provide recourse for consumers annoyed by excessive data collection, such as clogged mailboxes, without mandating separate data gathering that could create unintended security risks or profile consumers. Ultimately, the goal is to ensure honesty and balance between commerce and privacy through industry self-regulation and existing laws rather than strict mandates that could compromise security.

**2002** (13 statements, 116 words): The proposed legislation seeks to regulate online data practices by requiring companies to provide clear, conspicuous, and understandable privacy policies rather than relying on complex legalese or voluntary self-regulation. It aims to protect consumer data by establishing strong enforcement mechanisms to punish bad actors who mishandle information, addressing concerns that current fragmented laws and industry initiatives fail to control abuses like unauthorized data collection or the creation of third-party network nodes. The bill specifically targets the lack of consumer empowerment caused by unreadable notices and the potential for massive data breaches, such as the unauthorized dissemination of email addresses, by mandating a sensible federal initiative to ensure preemption protection and maintain consumer confidence in the internet.

**2003** (8 statements, 100 words): The discussed legislation seeks to establish baseline protections for consumer data privacy, specifically addressing the exposure of private materials such as health information to the general public. It aims to grant consumers control over their digital communications, including their inboxes, while distinguishing between responsible companies and non-compliant entities. The proposed measures target digital piracy of video content and content delivery television (DTV) through specific awareness acts. Furthermore, the legislation emphasizes the need to balance national security efforts, such as counter-terrorism, with the preservation of civil liberties and privacy rights, ensuring these concerns are addressed proactively rather than as an afterthought.

**2004** (4 statements, 117 words): The proposed legislation seeks to prohibit companies and organizations from secretly or deceptively monitoring consumer online activities without explicit knowledge or consent, specifically targeting misleading practices associated with spyware that threaten consumer confidence. It requires that directory information not reveal telephone numbers unless the consumer has opted in and explicitly requested the number, thereby regulating how personal contact details are disclosed. The bill aims to establish a general online privacy framework based on explicit permission, reflecting the value of privacy as an American principle. By mandating transparency regarding data collection and usage, the legislation regulates data practices involving surveillance and directory listings to ensure consumers are aware of and consent to the monitoring of their digital activities.

**2005** (2 statements, 89 words): The provided statements do not specify the legislation's requirements for companies, the data practices or kinds of data it regulates, or the specific actors it targets. Statement 83 advocates for a combination of technical solutions and tougher legislation to address issues similar to spam, while Statement 84 highlights long-standing committee concerns regarding security, privacy, and the shortfalls in protecting people amidst the growth of Internet commerce, encryption debates, and spam. Neither statement details the specific regulatory mechanisms, data categories, or organizational obligations mandated by the proposed or discussed legislation.

**2006** (8 statements, 108 words): The proposed legislation seeks to regulate how companies and organizations handle consumer data, specifically targeting the disclosure of personal information without consent. It aims to protect individuals from identity theft, stalking, and domestic violence by preventing the unauthorized sharing of private details such as phone records and call logs. A key provision requires that cell phone numbers be excluded from wireless directories unless the subscriber explicitly elects to be included. Furthermore, the law mandates that the Federal Communications Commission (FCC) actively enforce privacy rights and ensure that data practices do not expose consumers to harm from abusive spouses, ex-boyfriends, or opportunistic entities seeking to profit from privacy violations.

**2007** (1 statements, 101 words): The statements indicate that the proposed legislation seeks to regulate the sharing of consumer data, specifically highlighting social security numbers and medical information. It aims to protect the privacy and confidentiality of patient records held by medical professionals and organizations. The legislation addresses concerns regarding the potential for this information sharing to lead to discrimination in areas such as insurance insurability and hiring practices. Consequently, the rules would likely impose obligations on companies and other entities handling such sensitive data to prevent its misuse for discriminatory purposes, thereby safeguarding individuals from adverse outcomes based on their shared personal and medical records.

**2008** (6 statements, 119 words): The proposed legislation seeks to regulate companies and other organizations by mandating respect for consumer privacy protections, particularly regarding broadband services. It requires providers to ensure similar protection across all platforms and to protect sensitive information. Specifically, the law aims to ensure customers are informed about how their data is used, transferred, and sold, allowing them to make informed choices and control access to their information. It addresses data practices involving the collection, storage, and sale of identifiable information, requiring clear opt-in regimes or transparency about other data handling methods. While acknowledging the government's motivation, the legislation also considers the need to balance these privacy rules with operational efficiency in sectors like healthcare, where existing regulations may create inefficiencies.

**2010** (10 statements, 131 words): The proposed legislation seeks to regulate how companies and organizations handle consumer data, specifically addressing the collection, sharing, and disclosure of personal information. It aims to ensure uniformity in privacy notices, similar to credit card disclosures, requiring clear writing so consumers understand what data is collected. The law targets practices where data sharing discourages security or aids marketing without consumer awareness, such as barcodes revealing personal usage to vendors. It also addresses concerns regarding the distribution of private images and the need for sophisticated stalker laws, while acknowledging legitimate privacy concerns related to surveillance in public spaces like airports. Ultimately, the legislation attempts to balance consumer expectations of privacy with the necessity of information sharing and security, mandating that organizations provide transparency and protect sensitive data from unauthorized or misleading use.

**2011** (18 statements, 128 words): The proposed legislation seeks to establish a "commercial privacy bill of rights" that imposes clear rules on companies, including tech giants like Google, Apple, Facebook, Intel, eBay, Microsoft, and HP, as well as third-party aggregators. It requires these actors to stop consumers from acting as detectives by mandating that phone companies and aggregators crack down on data theft rather than leaving enforcement to individuals. The law aims to regulate specific data practices, such as the unauthorized mapping of consumer movements, the transfer of location data to ad networks without consent, and the collection of geolocation information, with particular emphasis on protecting children. It demands that organizations provide obvious privacy protections, ensure proper treatment of personal information, and engage constructively to modernize laws that balance consumer safety with innovation.

**2012** (7 statements, 135 words): The proposed legislation aims to regulate companies and organizations regarding consumer data privacy by requiring strong, consumer-focused protections for individuals in commerce. It seeks to address specific data practices such as the unauthorized invasion of personal space, inaccurate categorization of individuals, and the use of data for unwanted targeting or identity theft. The law would mandate that user activities within homes be encrypted for legal protection and would likely address the implementation of regulations like "Do Not Track." While some speakers express concern that overly restrictive rules might hinder innovation or require costly compliance, the core intent is to establish a legal framework where privacy experts agree that individual protection in commercial transactions is necessary. The legislation targets actors involved in commerce and private sector data handling to prevent reputational harm and unauthorized data usage.

**2013** (7 statements, 127 words): The discussed legislation seeks to regulate data practices and actors involved in online privacy, cyber intrusions, and information sharing. It addresses concerns regarding sensitive information leaking through "back door" cyber intrusions or "front door" channels with tacit approval from research centers, as well as the use of proprietary data. The proposed rules aim to protect civil liberties and personal privacy while managing legitimate fears in the healthcare field, specifically regarding HealthCare.gov. Furthermore, the legislation considers the monetization of the Internet through behavioral marketing and the costs associated with returning free content, while also encouraging companies to share threat information and best practices with each other and the government. Ultimately, the framework targets organizations like research centers and companies to ensure safer online environments and safeguarded personal data.

**2014** (5 statements, 135 words): The statements discuss legislation targeting commercial drone operators, law enforcement, and the Federal Aviation Administration (FAA). The proposed Drone Aircraft Privacy and Transparency Act requires commercial drone operators to disclose specific data practices, including what data is collected, how it is used, whether it will be sold, and when it will be deleted. It mandates that law enforcement obtain a warrant before using drones, except during emergency circumstances. Additionally, the FAA is required to create a publicly available website listing when and where drones fly. The legislation also addresses broader data security and breach notification through the Data Security and Breach Notification Act of 2013, though specific operational requirements for that act are not detailed in these statements. The primary focus is on regulating data collection transparency, usage disclosures, and warrant requirements for drone operations.

**2015** (18 statements, 134 words): The proposed legislation seeks to regulate data practices across various sectors, including healthcare, consumer tracking, and commercial drones. It requires organizations to implement best practices for securing large collections of personal information and mandates that the Administration prove it is protecting American privacy. Specifically, the Drone Aircraft Privacy and Transparency Act demands that commercial drone operators disclose how collected data is used, whether it is sold, and its deletion status. It also requires law enforcement to obtain warrants before using drones, except in emergencies, and mandates the FAA to publish a website listing drone flight locations and times. Additionally, the legislation aims to establish strong federal data protection and notification requirements for data breaches, preempts inconsistent state laws, and calls for rules to protect driver privacy regarding data gathered by connected devices and vehicles.

**2016** (21 statements, 133 words): The discussed legislation seeks to regulate data practices by requiring enforceable rules, mandatory minimum privacy protections, and explicit consumer consent for the collection, transmission, retention, and use of personal information. It specifically targets driving data, banking information, customer proprietary network information, and data held by the federal government. The proposed regulations aim to prevent the use of personal data for advertising without clear opt-in, ensure robust data security measures, and mandate honest disclosures by companies and entities that collect or store consumer data. Key actors addressed include the Federal Trade Commission, the Federal Communications Commission, the Federal Deposit Insurance Corporation, the Environmental Protection Agency, and various private organizations such as retailers, insurers, and hospitals. The legislation also seeks to clarify legal standards to prevent inconsistent privacy protections based on how services are classified.

**2017** (11 statements, 132 words): The legislation discussed seeks to regulate data practices by imposing rigorous security rules and accountability measures on private sector actors, particularly broadband providers and companies handling consumer information. It specifically targets the collection, sharing, and use of sensitive personal data, including identity information and data gathered from devices like "computers on wheels" as individuals move across the country. The proposed rules aim to prevent the sale and sharing of consumer data without baseline protections, ensuring that companies cannot exploit vast amounts of gathered information for purposes other than those explicitly permitted. The legislation also addresses the use of collected information for regulatory oversight, though some statements express concern regarding blanket prohibitions on such oversight. Ultimately, the goal is to place consumers in control of their sensitive information and prevent future data breaches.

**2018** (44 statements, 129 words): The proposed legislation seeks to regulate data collection, retention, sale, and usage practices across the internet ecosystem, specifically targeting social media companies, tech industry actors, drone operators, and federal agencies like the FTC. It requires companies to provide transparent, clear information to consumers rather than opaque policies, ensuring individuals can make informed decisions about sharing their data. The law aims to establish uniform privacy protections regardless of location, incorporating standards similar to the EU's GDPR and California's CCPA. Key requirements include granting consumers ownership rights or opt-out capabilities, mandating breach notifications within 72 hours, and enforcing "privacy by design" principles. The legislation specifically addresses personally identifiable information, including content, contact lists, location, and device data, while demanding that federal agencies actively monitor compliance rather than adopting a hands-off approach.

**2019** (40 statements, 143 words): The proposed legislation seeks to establish a federal framework that restricts the monetization of personal data and mandates robust cybersecurity regimes to protect consumers, industries, and government institutions. It aims to regulate companies' data practices by requiring transparency, eliminating restrictive confidentiality statements, and ensuring consumers have the right to control their private information. The law targets actors who possess vast amounts of data, such as big tech incumbents, while also protecting small businesses and startups from overly burdensome regulations. Specific provisions address the collection, retention, and re-identification of consumer data, particularly concerning seniors, and require clear statutory requirements with flexible rulemaking authorities for enforcement agencies like the FTC. The legislation also emphasizes the need for strong enforcement mechanisms and civil rights protections to ensure privacy violations are detected and remedied, while fostering innovation through a baseline of privacy rights rather than stifling competition.

**2020** (6 statements, 108 words): The proposed legislation seeks to establish a uniform national standard for data security and privacy, aiming to protect consumer information through stronger protections. It specifically targets the regulation of online content moderation decisions and algorithms that amplify or suppress information, requiring transparency for users, regulators, and the public regarding these processes. The bills also address the need for meaningful insight into how platforms manage content, while acknowledging the importance of resolving issues related to data sharing mechanisms like Privacy Shield for small and medium-sized enterprises. Ultimately, the legislation aims to provide a framework for data security and consumer transparency across various digital platforms and organizations handling user data.

**2021** (62 statements, 120 words): The proposed legislation seeks to regulate big tech platforms, cable companies, and app-based service providers by establishing comprehensive federal privacy laws and empowering the FTC with new resources and enforcement authority. It requires companies to obtain expressed consent before using personal information, provide transparency regarding algorithmic content selection, and prevent the use of data to harm individuals or discriminate against vulnerable populations. Specific provisions target the collection of health data from wearables, voice recordings of children, and tax-related information, aiming to address data security violations and curb anti-competitive practices. The laws also mandate public disclosures about potential harms and seek to reform Section 230 to increase accountability for illegal content while ensuring startups are not stifled by uncalibrated security requirements.

**2022** (1 statements, 94 words): The statement indicates that the proposed uniform national privacy legislation aims to protect consumers from identity theft and the misuse of their personal information, particularly in the context of COVID-related scams. While the text does not specify which exact data practices, kinds of data, or specific organizational actors must be regulated, it establishes a clear mandate for companies and other entities to safeguard consumer personal information against unauthorized use and fraud. The legislation seeks to create a consistent national standard to prevent the exploitation of individual data, addressing the vulnerabilities exposed by recent scams.

**2023** (10 statements, 124 words): The discussed legislation primarily seeks to establish a federal privacy standard and empower the Federal Trade Commission to safeguard personal information, aiming to prevent discriminatory actions and protect consumers within the AI-enabled economy. It specifically targets data practices involving the sharing of information, emphasizing the need for privacy-enhancing technologies to support AI training systems while ensuring responsible government coordination. The proposed laws address concerns regarding foreign entities accessing American company data and seek to regulate data use to mitigate national security risks and prevent harm caused by substantial data direction. Ultimately, the legislation aims to create a comprehensive framework that balances consumer empowerment with the growth of modern technology, focusing on preventing mistakes and ensuring legal and security compliance for organizations handling sensitive data.

**2024** (9 statements, 122 words): The proposed legislation seeks to establish a national standard requiring companies to obtain consumer approval before buying or selling private data, thereby restricting the sale of sensitive information without consent. It aims to regulate tech companies by mandating that they implement these laws and stop data interference. The legislation targets specific data practices, including the commercialization of private data and algorithmic regulation, intending to empower individuals with control over their information while ensuring legal certainty for businesses. However, one statement argues that the specific draft being discussed delegates too much power to the FTC and focuses on algorithmic regulation under the guise of civil rights rather than personal privacy, suggesting it may harm free speech and benefit large corporations over small businesses.

**2025** (6 statements, 124 words): The proposed legislation seeks to establish federal online consumer privacy standards, granting individuals the ability to set data "firewalls." It aims to prohibit sweeping data privacy and security mandates imposed without Congressional approval and addresses the lack of protections resulting from specific organizational activities. The law targets the rapid adoption of artificial intelligence, requiring the highest standards for data protection and governance to prevent exposure of American information. Furthermore, it mandates that companies ensure data is shared in a timely manner to facilitate informed policy decisions and industry dominance, while also considering these privacy measures alongside actual AI legislation. The regulations focus on data practices regarding collection, protection, and sharing, specifically concerning data held by organizations involved in AI development and general online activities.

## Step 6. Show the final policy summary

This is the text T3.5.7 embeds as the third topic vector.


In [6]:
display(Markdown("**Policy summary, Topic 68**\n\n" + policy_summary))


**Policy summary, Topic 68**

The legislation imposes rigorous obligations on tech giants, broadband providers, healthcare organizations, drone operators, and financial institutions to secure consumer data and prevent its unauthorized sale or misuse. It grants individuals specific rights, including the ability to opt in or out of data collection, demand clear and conspicuous privacy policies, control their digital communications, and establish data "firewalls." Companies are required to implement "privacy by design," provide breach notifications within 72 hours, and ensure that sensitive information such as health records, social security numbers, and geolocation data is protected from discrimination and identity theft.

The regulatory scope targets a wide array of data practices, ranging from the deceptive monitoring of online activities and the unauthorized mapping of consumer movements to the commercialization of private data and the use of algorithms to amplify or suppress content. Specific actors include research centers, third-party aggregators, and federal agencies, with particular emphasis on preventing the export of products and the sharing of personal information without consent. The framework addresses diverse data types, including directory information, voice recordings, and data gathered by connected vehicles, aiming to balance national security interests with civil liberties. Ultimately, the legislation seeks to replace fragmented standards with a uniform national baseline that mandates transparency, enforces accountability, and empowers consumers to make informed choices regarding their personal information in the digital economy.

## Step 7. Trace the summary's details to the statements

Count, for each listed detail, how many source statements contain it. A detail with zero matches did not come from the statements.


In [7]:
statements = pd.read_csv(STATEMENTS_FILE)
trace_rows = []
for term in TRACE_TERMS:
    assert term.lower() in policy_summary.lower() or term in ("opt in", "opt out", "federal agenc"), f"{term!r} is not in the summary"
    matches = statements.loc[statements["text"].str.contains(term, case=False, regex=False)]
    trace_rows.append({
        "detail": term,
        "statements": len(matches),
        "first_year": int(matches["hearing_year"].min()) if len(matches) else None,
        "example": matches["text"].iloc[0][:160] if len(matches) else "",
    })
trace = pd.DataFrame(trace_rows)
unsourced = trace.loc[trace["statements"].eq(0), "detail"].tolist()
print(f"Details traced: {len(trace)}; with at least one source statement: {int(trace['statements'].gt(0).sum())}; unsourced: {unsourced}")
pd.set_option("display.max_colwidth", 160)
display(trace)


Details traced: 26; with at least one source statement: 21; unsourced: ['financial institution', 'clear and conspicuous', 'health record', 'directory information', 'connected vehicle']


,detail,statements,first_year,example
0,broadband,9,2008.0,"And as broadband service is so vital to the American people and to our communities, we want to make sure that providers are respecting the privacy protectio..."
1,healthcare,6,2013.0,"So while there can be legitimate concerns about the privacy in the health care field, HealthCare.gov should not be the case of any exceptional fears in that..."
2,drone,6,2014.0,"So that's why in the House last Congress, I introduced a bill and I've introduced the same bill in the Senate, by the way, the Drone Aircraft Privacy and Tr..."
3,financial institution,0,NaN,
4,opt in,1,2001.0,"I am, as I look at this question of privacy, more and more pro-choice here in terms of the consumer or the individual. I think that is where the question of..."
5,opt out,1,2001.0,"I am, as I look at this question of privacy, more and more pro-choice here in terms of the consumer or the individual. I think that is where the question of..."
6,clear and conspicuous,0,NaN,
7,firewall,1,2025.0,"Online privacy is something as we have worked on AI, we have heard from so many innovators, it is imperative to pass an online consumer privacy standard so ..."
8,privacy by design,1,2018.0,There needs to be privacy by design. Call it a privacy bill of rights. Call it a mandatory standard or a floor.
9,72 hours,1,2018.0,"And Senator Kennedy and I introduced--there are a number of bills out there, but we introduced a bipartisan bill that would make privacy disclosures more tr..."


## Step 8. Write the check file

A readable record: run facts, the call table, the trace, every per-year summary, and the final summary.


In [8]:
check_file = OUTPUT_FOLDER / "policy_summary_check.md"
check_text = (
    "# Policy summary check, Topic 68\n\n"
    f"Run {meta['run_started']} to {meta['run_finished']} on {meta['gpu']}; model {Path(meta['model']).parent.parent.name}; "
    f"generation {meta['generation']}, thinking {meta['enable_thinking']}, seed {meta['seed']}; "
    f"{meta['n_map_calls']} map calls and {len(meta['reduce_rounds'])} reduce round(s); no call hit the output cap.\n\n"
    f"Final summary: {meta['final_n_words']} words, {meta['final_n_bge_tokens']} bge tokens (limit {MAX_BGE_TOKENS}).\n\n"
    "## Calls\n\n" + call_table.to_markdown(index=False) + "\n\n"
    "## Detail trace\n\n"
    f"Details with at least one source statement: {int(trace['statements'].gt(0).sum())} of {len(trace)}; unsourced: {unsourced}.\n\n"
    + trace[["detail", "statements", "first_year"]].to_markdown(index=False) + "\n\n"
    f"Years with at most two statements: {small_years['hearing_year'].tolist()}, summary lengths {small_years['n_words'].tolist()} words.\n\n"
    "## Per-year summaries\n\n" + yearly_markdown + "\n\n"
    "## Final policy summary\n\n" + policy_summary + "\n"
)
check_file.write_text(check_text, encoding="utf-8")
print(f"Saved: {check_file}")


Saved: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.5.6/policy_summary_check.md
